# Individual Conditional Expectations (ICE)
ICE curves show how each individual example's prediction changes with a feature. When ICE curves diverge, it indicates feature interactions.

In [ ]:
import skexplain
import plotting_config

In [ ]:
# Load the training data and pre-fit models
estimators = skexplain.load_models()
X, y = skexplain.load_data()
X = X.astype({'urban': 'category', 'rural': 'category'})

# Use the Random Forest model only
explainer = skexplain.ExplainToolkit(estimators[0], X=X, y=y)

explainer.set_plotting_config(
    display_feature_names=plotting_config.display_feature_names,
    display_units=plotting_config.display_units,
    feature_colors=plotting_config.color_dict,
)

## Computing ICE Curves

ICE curves are computed with `explainer.ice()`. The `subsample` argument controls how many individual curves are drawn. Plotting more than 200 curves makes the plot hard to read.

In [ ]:
features = ['sfc_temp', 'temp2m', 'dwpt2m', 'sfcT_hrs_bl_frez']

ice_ds = explainer.ice(
    features=features,
    subsample=200,
    n_jobs=4,
    n_bins=20,
)

In [ ]:
# Compute ALE for the same features (used as the overlay mean line)
ale_1d_ds = explainer.ale(
    features=features,
    n_bootstrap=1,
    subsample=10000,
    n_jobs=1,
    n_bins=20,
)

## Plotting ALE with ICE Overlay

Pass the ICE dataset to `plot_ale` via the `ice_curves` argument. The bold line is the ALE (mean effect) and the thin lines are individual ICE curves. Spread in the ICE curves indicates feature interactions.

In [ ]:
fig, axes = explainer.plot_ale(
    ale=ale_1d_ds,
    ice_curves=ice_ds,
)

## Color-Coded ICE Curves

Color-coding ICE curves by another feature's value reveals interactions between features. Use the `color_by` argument to specify which feature to color by.

In [ ]:
fig, axes = explainer.plot_ale(
    ale=ale_1d_ds,
    features=features,
    ice_curves=ice_ds,
    color_by='temp2m',
    figsize=(10, 6),
    wspace=0.25,
)

When the colored ICE curves separate into distinct bands (e.g., warm colors trending differently than cool colors), it indicates that the feature being plotted interacts with the `color_by` feature. If all colored lines follow the same trend regardless of color, there is little interaction between the two features.

ICE curves are a permutation-based method and assume feature independence, so correlated features can muddle the interpretation. Use them as an exploratory tool alongside ALE and 2D ALE for a more complete picture.